# Movies Project Starter

Use this starter notebook for the BAN 6003 final project option: **The Movies Dataset / MovieLens ratings**.

Run cells from top to bottom first. Then replace starter notes with your own project decisions, checks, and interpretation.

Start each milestone by reading the matching Canvas milestone assignment page. The Canvas page tells you exactly what evidence and submission items are required; this notebook gives you starter spaces to build that work.



## Before You Start: Key Project Hints

This dataset mixes movie-level tables and rating-event-level data. The key skill is controlling granularity.

- A common final ABT is **one row per movie**.
- Do **not** directly merge raw `ratings` into `movies` and assume the result is still one row per movie.
- Aggregate ratings to `movieId` first, then merge the rating summary into the movie table.
- Use `movieId` for ratings and `tmdbId` for curated credits/keywords.
- Average rating is unstable for movies with very few ratings. Use a minimum rating-count threshold before modeling.
- Be careful about leakage: post-release fields such as revenue, popularity, vote count, or rating count may not be valid if your question is pre-release prediction.

## Setup

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)


## Load Data

Read each provided table. Keep the original row meaning in mind before joining tables.

In [2]:
movies = pd.read_csv(DATA_DIR / 'movies_metadata.csv')
ratings = pd.read_csv(DATA_DIR / 'ratings.csv', parse_dates=['rating_datetime'])
links = pd.read_csv(DATA_DIR / 'links.csv')
credits = pd.read_csv(DATA_DIR / 'credits_curated.csv')
keywords = pd.read_csv(DATA_DIR / 'keywords_curated.csv')

tables = {
    'movies': movies,
    'ratings': ratings,
    'links': links,
    'credits': credits,
    'keywords': keywords,
}

## Basic Data Profile

Use this section for your first pass through row counts, columns, data types, missingness, duplicates, and suspicious values.

In [3]:
for name, df in tables.items():
    print(f'\n{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head(3))


movies: 9,082 rows x 21 columns


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995



ratings: 99,810 rows x 5 columns


,userId,movieId,rating,timestamp,rating_datetime
0,1,31,2.5,1260759144,2009-12-14 02:52:24
1,1,1029,3.0,1260759179,2009-12-14 02:52:59
2,1,1061,3.0,1260759182,2009-12-14 02:53:02



links: 9,125 rows x 3 columns


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0



credits: 9,082 rows x 3 columns


,tmdbId,director,top_cast
0,862,John Lasseter,Tom Hanks|Tim Allen|Don Rickles
1,8844,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst
2,15602,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret



keywords: 9,082 rows x 2 columns


,tmdbId,keywords_list
0,862,jealousy|toy|boy|friendship|friends|rivalry|bo...
1,8844,board game|disappearance|based on children's b...
2,15602,fishing|best friend|duringcreditsstinger|old men


In [4]:
profile_rows = []
for name, df in tables.items():
    profile_rows.append({
        'table': name,
        'rows': len(df),
        'columns': df.shape[1],
        'duplicate_rows': int(df.duplicated().sum()),
        'missing_cells': int(df.isna().sum().sum()),
    })
pd.DataFrame(profile_rows)

,table,rows,columns,duplicate_rows,missing_cells
0,movies,9082,21,0,72
1,ratings,99810,5,0,0
2,links,9125,3,0,13
3,credits,9082,3,0,114
4,keywords,9082,2,0,761


### Key and Granularity Audit

Use the expected key for each table before planning joins. `ratings` is the exception: its row meaning is a user-movie rating event, so check the combined `userId`-`movieId` key rather than expecting `movieId` alone to be unique. `links` is a crosswalk; because `movies` already contains both IDs, use it for validation unless your analysis needs one of its fields.


In [5]:
key_audit = pd.DataFrame([
    {"table": "movies", "expected_key": "movieId", "duplicate_keys": movies["movieId"].duplicated().sum(), "missing_key_values": movies["movieId"].isna().sum()},
    {"table": "movies", "expected_key": "tmdbId", "duplicate_keys": movies["tmdbId"].duplicated().sum(), "missing_key_values": movies["tmdbId"].isna().sum()},
    {"table": "ratings", "expected_key": "userId + movieId", "duplicate_keys": ratings[["userId", "movieId"]].duplicated().sum(), "missing_key_values": ratings[["userId", "movieId"]].isna().sum().sum()},
    {"table": "links", "expected_key": "movieId", "duplicate_keys": links["movieId"].duplicated().sum(), "missing_key_values": links["movieId"].isna().sum()},
    {"table": "credits", "expected_key": "tmdbId", "duplicate_keys": credits["tmdbId"].duplicated().sum(), "missing_key_values": credits["tmdbId"].isna().sum()},
    {"table": "keywords", "expected_key": "tmdbId", "duplicate_keys": keywords["tmdbId"].duplicated().sum(), "missing_key_values": keywords["tmdbId"].isna().sum()},
])

key_audit


,table,expected_key,duplicate_keys,missing_key_values
0,movies,movieId,0,0
1,movies,tmdbId,0,0
2,ratings,userId + movieId,0,0
3,links,movieId,0,0
4,credits,tmdbId,0,0
5,keywords,tmdbId,0,0


## Milestone 1 Starter: Initial Profile and Cleaning Plan

Complete this by the first project milestone.

Write notes on:

- Business problem and decision context
- Raw tables and row meaning
- Initial row and column counts
- Data type issues
- Missing values and duplicates
- Suspicious values
- Proposed target/outcome
- Early ethics or governance concerns

Important Movies notes:

- `movies` is movie-level, while `ratings` is user-movie rating-event-level.
- `credits` and `keywords` are curated movie-level tables connected by `tmdbId`.
- Budget and revenue may contain zero or missing-like values; describe this before using them.
- Decide whether your project is about audience ratings, catalog strategy, revenue, popularity, or genre/content patterns.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 1 written workspace

Use this cell to draft the narrative required by the Canvas Milestone 1 page.

- Business or research question:
    - What factors are associated with higher user ratings?
- Why this question matters:
    - If there are certain factors in movies that lead to a higher user rating, this can create a prediction model for production companies to use.
- Tables/files used:
    - ratings.csv and movies_metadata.csv
- Row meaning of each important table:
    - For ratings, one row contains the rating of a movie and when it was rated. For movies_metadata, one row contains the information of a movie. This includes if it has been released, the genre_list, the primary genre of the movie, and even the budget and its revenue.
- Early data quality issues:
    - A potential issue that may arise could be ratings that are not valid, such as higher than a 5.0.
- Initial cleaning plan:
    - Confirm the row count and column counts match the tables. Confirm if there are any values missing and/or duplicate values.
- Possible target/outcome: 
    - The target is to understand if you can predict how well a movie will perform based off certain qualities/factors of the movie. This can be asked in a way such as "What are rated higher, action movies or family movies?"
- Ethics, privacy, or governance concerns:
    - As far as I can tell, there are no privacy concerns. An ethical concern could be that if a prediction model can be created from the data that has been cleaned, movie production companies could simply create certain movies under certain genres that generate higher ratings. This could create a dilemma where only, for example. action and adventure movies are being made whereas no family or comdey movies are being produced.



In [ ]:
# Milestone 1 code workspace
# Add your additional profiling checks here.
# Examples: missingness by important columns, duplicate key checks, suspicious value checks.

raw_path = Path("../data/ratings.csv")
ratings_raw = pd.read_csv(raw_path)
ratings_raw.head()

ratings_raw.info()

row_count, column_count = ratings_raw.shape
rating_missing_count = ratings_raw["rating"].isna().sum()
duplicate_user_movie_count = ratings_raw.duplicated(subset=["userId", "movieId"]).sum()

print(f"Rows: {row_count}")
print(f"Columns: {column_count}")
print(f"Missing rating values: {rating_missing_count}")
print(f"Duplicate user-movie pairs: {duplicate_user_movie_count}")

<class 'pandas.DataFrame'>
RangeIndex: 99810 entries, 0 to 99809
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   userId           99810 non-null  int64  
 1   movieId          99810 non-null  int64  
 2   rating           99810 non-null  float64
 3   timestamp        99810 non-null  int64  
 4   rating_datetime  99810 non-null  str    
dtypes: float64(1), int64(3), str(1)
memory usage: 3.8 MB
Rows: 99810
Columns: 5
Missing rating values: 0
Duplicate user-movie pairs: 0


In [75]:
raw_path_2 = Path("../data/movies_metadata.csv")
metadata_raw = pd.read_csv(raw_path_2)
metadata_raw.head()

metadata_raw.info()

row_count, column_count = metadata_raw.shape
rating_missing_count = metadata_raw["movieId"].isna().sum()
duplicate_user_movie_count = metadata_raw.duplicated(subset=["tmdbId", "imdb_id"]).sum()

print(f"Rows: {row_count}")
print(f"Columns: {column_count}")
print(f"Missing rating values: {rating_missing_count}")
print(f"Duplicate user-movie pairs: {duplicate_user_movie_count}")

<class 'pandas.DataFrame'>
RangeIndex: 9082 entries, 0 to 9081
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   movieId               9082 non-null   int64  
 1   tmdbId                9082 non-null   int64  
 2   imdb_id               9082 non-null   str    
 3   title                 9082 non-null   str    
 4   original_title        9082 non-null   str    
 5   original_language     9082 non-null   str    
 6   release_date          9082 non-null   str    
 7   runtime               9082 non-null   float64
 8   budget                9082 non-null   int64  
 9   revenue               9082 non-null   float64
 10  popularity            9082 non-null   float64
 11  vote_average          9082 non-null   float64
 12  vote_count            9082 non-null   float64
 13  genres                9082 non-null   str    
 14  production_companies  9082 non-null   str    
 15  production_countries  9082 non-n

In [76]:
ratings_raw.isna().sum()

userId             0
movieId            0
rating             0
timestamp          0
rating_datetime    0
dtype: int64

In [77]:
metadata_raw.isna().sum()

movieId                  0
tmdbId                   0
imdb_id                  0
title                    0
original_title           0
original_language        0
release_date             0
runtime                  0
budget                   0
revenue                  0
popularity               0
vote_average             0
vote_count               0
genres                   0
production_companies     0
production_countries     0
adult                    0
status                   2
genres_list             35
primary_genre           35
release_year             0
dtype: int64

In [ ]:
ratings = ratings_raw.copy()
ratings = ratings[["userId", "movieId", "rating", "timestamp", "rating_datetime"]].head()

In [181]:
user_ratings = ratings.query(
    "rating > 3.0",
    engine="python",
)

user_ratings = user_ratings[[
    "userId", "movieId", "rating", "timestamp", "rating_datetime"
]]
user_ratings = user_ratings.rename(columns={
    "userId": "User_ID",
    "movieId": "Movie_ID"
})
user_ratings = user_ratings.sort_values(
    ["rating", "Movie_ID"], ascending=[False,True]
).reset_index(drop=True)

user_ratings.head()

,User_ID,Movie_ID,rating,timestamp,rating_datetime
0,1,1172,4.0,1260759205,2009-12-14 02:53:25


In [169]:
metadata = metadata_raw.copy()
metadata = metadata[["movieId", "tmdbId", "title", "vote_count","genres_list","primary_genre"]].head()

In [159]:
metadata_vote = metadata.query(
    "vote_count > 0"
)
metadata_vote.sort_values("vote_count", ascending=False).head()

,movieId,tmdbId,title,vote_count,genres_list,primary_genre
0,1,862,Toy Story,5415.0,Animation|Comedy|Family,Animation
1,2,8844,Jumanji,2413.0,Adventure|Fantasy|Family,Adventure
4,5,11862,Father of the Bride Part II,173.0,Comedy,Comedy
2,3,15602,Grumpier Old Men,92.0,Romance|Comedy,Romance
3,4,31357,Waiting to Exhale,34.0,Comedy|Drama|Romance,Comedy


In [177]:
vote_query = metadata.query(
    "vote_count > 100",
    engine="python",
)

vote_query = vote_query[[
    "movieId", "tmdbId", "title", "vote_count","genres_list","primary_genre"
]]
vote_query = vote_query.rename(columns={
    "movieId": "Movie_ID",
    "tmdbId": "TMDb_ID",
    "title": "Title"
})
vote_query = vote_query.sort_values(
    ["vote_count", "Movie_ID"], ascending=[True,True]
).reset_index(drop=True)

vote_query.head()

,Movie_ID,TMDb_ID,Title,vote_count,genres_list,primary_genre
0,5,11862,Father of the Bride Part II,173.0,Comedy,Comedy
1,2,8844,Jumanji,2413.0,Adventure|Fantasy|Family,Adventure
2,1,862,Toy Story,5415.0,Animation|Comedy|Family,Animation


In [168]:
# Milestone 1 starter checks
for name, df in tables.items():
    print(f'\n{name}')
    display(df.dtypes.to_frame('dtype').head(20))
    display(df.isna().mean().sort_values(ascending=False).head(10).to_frame('missing_rate'))


movies


,dtype
movieId,int64
tmdbId,int64
imdb_id,str
title,str
original_title,str
original_language,str
release_date,str
runtime,float64
budget,int64
revenue,float64


,missing_rate
genres_list,0.003854
primary_genre,0.003854
status,0.000220
tmdbId,0.000000
movieId,0.000000
original_title,0.000000
title,0.000000
imdb_id,0.000000
original_language,0.000000
revenue,0.000000



ratings


,dtype
userId,int64
movieId,int64
rating,float64
timestamp,int64
rating_datetime,datetime64[us]


,missing_rate
userId,0.0
movieId,0.0
rating,0.0
timestamp,0.0
rating_datetime,0.0


... (5 outputs left out)

,missing_rate
top_cast,0.009910
director,0.002643
tmdbId,0.000000



keywords


,dtype
tmdbId,int64
keywords_list,str


,missing_rate
keywords_list,0.083792
tmdbId,0.000000


## Milestone 2 Starter: Integration, Transformation, and Preliminary ABT

Build a preliminary ABT. State exactly what one row means, what keys define one row, and what tables were joined or aggregated.

Critical integration reminder:

Before creating a movie-level ABT, aggregate ratings. Examples include:

- average rating by `movieId`
- rating count by `movieId`
- rating standard deviation by `movieId`
- first and last rating date by `movieId`
- distinct user count by `movieId`

After merging, validate that `movieId` is not duplicated in the ABT.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 2 written workspace

Use this cell to document the ABT design required by the Canvas Milestone 2 page.

- Final or preliminary unit of analysis: 
    - one row per movie
- Primary key for one row:
    - The primary key is movieId.
- Tables aggregated before joining: 
    - The table I aggregated before joining is ratings.csv
- Tables joined directly:
    - The table that I joined directly is movies_metadata.csv
- Row-count validation after each major merge: 
    - Validation was successful. All rows and columns matched, as well as those that fit the parameters given.
- New variables created:
    - New variables include title and voting_count.
- Remaining data quality concerns:
    - There could be outliers still in the space, like movies with much lower/higher vote counts even after putting a limit. I could be missing something that would help with the business question previously stated.



In [7]:
# Milestone 2 code workspace
# Build or revise your ABT here.
# Add row-count checks and duplicate-key checks after each major merge.

import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Find the top folder of the repository (the folder that contains README.md).
repo_root = Path.cwd()
if not (repo_root / "README.md").exists() and (repo_root.parent / "README.md").exists():
    repo_root = repo_root.parent

# Add an empty PROJECT_DATABASE_URL line to .env the first time. Git ignores .env.
env_path = repo_root / ".env"
env_text = env_path.read_text() if env_path.exists() else ""
if "PROJECT_DATABASE_URL" not in env_text:
    if env_text and not env_text.endswith("\n"):
        env_text += "\n"
    env_text += ("# Your own Neon database (Path A). Paste your connection string after the = sign.\n"
                 "PROJECT_DATABASE_URL=\n")
    env_path.write_text(env_text)
    print("Added an empty PROJECT_DATABASE_URL line to .env")

load_dotenv(env_path, override=True)
project_url = os.getenv("PROJECT_DATABASE_URL")

# Saved query results go here, so the notebook also runs without your .env (see Section 3.4).
SQL_RESULTS_DIR = repo_root / "outputs" / "sql_results"
SQL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if project_url:
    project_engine = create_engine(project_url)
    with project_engine.connect() as conn:
        conn.execute(text("SELECT 1;"))
    print("Connected to your own database. For safety, we do not print the connection string.")
else:
    project_engine = None
    print("PROJECT_DATABASE_URL is empty, so the database steps will be skipped.")
    print("Path A: open .env, paste your own Neon connection string after PROJECT_DATABASE_URL=, save, and run this cell again.")

Connected to your own database. For safety, we do not print the connection string.


In [8]:
def upload_table(source, table_name):
    """Upload a CSV file path or a DataFrame to your own database and check the row count."""
    if project_engine is None:
        print(f"Skipped uploading {table_name}: no PROJECT_DATABASE_URL in .env.")
        return
    df = source if isinstance(source, pd.DataFrame) else pd.read_csv(source, low_memory=False)
    df.to_sql(table_name, project_engine, if_exists="replace", index=False, chunksize=1000, method="multi")
    with project_engine.connect() as conn:
        db_rows = conn.execute(text(f"SELECT COUNT(*) FROM {table_name};")).scalar()
    status = "OK" if db_rows == len(df) else "CHECK: the row counts differ"
    print(f"{table_name}: {len(df):,} rows in Python, {db_rows:,} rows in the database. {status}")


# Fill in your own files. One line per table: upload_table(path_or_dataframe, "table_name")
# In your project, use your data files, for example:
#   upload_table(repo_root / "data" / "ratings.csv", "ratings")
#   upload_table(abt, "movie_abt")      # a DataFrame you built in Pandas
upload_table(repo_root / "data" / "ratings.csv", "ratings")
upload_table(repo_root / "data" / "movies_metadata.csv", "metadata")

ratings: 99,810 rows in Python, 99,810 rows in the database. OK
metadata: 9,082 rows in Python, 9,082 rows in the database. OK


In [31]:
def run_project_sql(query, result_name):
    """Run SQL on your own database and save the result. Without a connection, show the saved result."""
    result_path = SQL_RESULTS_DIR / f"{result_name}.csv"
    if project_engine is not None:
        result = pd.read_sql(text(query), project_engine)
        result.to_csv(result_path, index=False)
        print(f"Ran on your database. Saved the result to {result_path.relative_to(repo_root)}")
        return result
    if result_path.exists():
        print(f"No database connection here, so this is the saved result from {result_path.relative_to(repo_root)}")
        return pd.read_csv(result_path)
    print(f"Skipped {result_name}: no PROJECT_DATABASE_URL and no saved result yet.")
    return pd.DataFrame()

In [38]:
## this is the actual ABT space
## this might be scuffed 

query = '''
SELECT
    r.userId,
    r.movieId,
    r.rating,
    m.title AS title
    m.vote_count AS vote_count
FROM ratings r
JOIN metadata m
    ON r.movieId = m.movieId
WHERE m.vote_count > 100
LIMIT 10;
'''

run_query = (query, "ratings_with_titles")



In [43]:
query = '''
WITH user_movies AS (
    SELECT
        userId,
        movieId,
        COUNT(*) AS rating,
        ROUND(AVG(rating)::numeric, 2)::float AS avg_rating,
    FROM ratings
    GROUP BY userId, movieId
)
SELECT
    um.*,
    m.title AS title
    m.vote_count AS vote_count
FROM user_movies AS um
LEFT JOIN metadata AS m
    ON um.movieId = m.movieId
WHERE m.vote_count > 100
ORDER BY um.userId, um.movieId;
'''

run_query = (query, "ratings_with_titles_and_avg_rating")

In [44]:
rating_summary = (
    ratings.groupby("movieId")
    .agg(
        user_rating_mean=("rating", "mean"),
        user_rating_count=("rating", "size"),
        user_rating_std=("rating", "std"),
        first_rating_date=("rating_datetime", "min"),
        last_rating_date=("rating_datetime", "max")
    )
    .reset_index()
)

abt = (
    movies
    .merge(rating_summary, on="movieId", how="left", validate="one_to_one")
    .merge(credits, on="tmdbId", how="left", validate="one_to_one")
    .merge(keywords, on="tmdbId", how="left", validate="one_to_one")
)

abt["has_revenue"] = (pd.to_numeric(abt["revenue"], errors="coerce") > 0).astype("Int64")
abt["rating_target_eligible"] = abt["user_rating_count"].ge(20)
abt["high_user_rating"] = (
    abt["user_rating_mean"].ge(3.75)
    .where(abt["rating_target_eligible"])
    .astype("Int64")
)


In [45]:
abt_validation = pd.Series({
    "movie_rows_before_merges": len(movies),
    "abt_rows_after_merges": len(abt),
    "duplicate_movie_ids": abt["movieId"].duplicated().sum(),
    "duplicate_tmdb_ids": abt["tmdbId"].duplicated().sum(),
    "movies_with_rating_summary": abt["user_rating_count"].notna().sum(),
    "movies_eligible_for_rating_target": abt["high_user_rating"].notna().sum(),
})

display(abt_validation)
display(abt.head())
display(abt.isna().mean().sort_values(ascending=False).head(15).to_frame("missing_rate"))


movie_rows_before_merges             9082
abt_rows_after_merges                9082
duplicate_movie_ids                     0
duplicate_tmdb_ids                      0
movies_with_rating_summary           9025
movies_eligible_for_rating_target    1300
dtype: int64

,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year,user_rating_mean,user_rating_count,user_rating_std,first_rating_date,last_rating_date,director,top_cast,keywords_list,has_revenue,rating_target_eligible,high_user_rating
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995,3.872470,247.0,0.958981,1996-03-30 19:00:13,2016-10-06 19:55:11,John Lasseter,Tom Hanks|Tim Allen|Don Rickles,jealousy|toy|boy|friendship|friends|rivalry|bo...,1,True,1
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995,3.401869,107.0,0.880714,1996-03-30 19:12:30,2016-08-01 17:42:33,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst,board game|disappearance|based on children's b...,1,True,0
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995,3.161017,59.0,1.150115,1996-06-05 06:19:04,2016-08-16 22:07:21,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret,fishing|best friend|duringcreditsstinger|old men,0,True,0
3,4,31357,tt0114885,Waiting to Exhale,Waiting to Exhale,en,1995-12-22,127.0,16000000,81452156.0,3.859495,6.1,34.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",[{'name': 'Twentieth Century Fox Film Corporat...,"[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy|Drama|Romance,Comedy,1995,2.384615,13.0,0.938835,1996-06-10 16:45:35,2004-07-27 06:14:12,Forest Whitaker,Whitney Houston|Angela Bassett|Loretta Devine,based on novel|interracial relationship|single...,1,False,<NA>
4,5,11862,tt0113041,Father of the Bride Part II,Father of the Bride Part II,en,1995-02-10,106.0,0,76578911.0,8.387519,5.7,173.0,"[{'id': 35, 'name': 'Comedy'}]","[{'name': 'Sandollar Productions', 'id': 5842}...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy,Comedy,1995,3.267857,56.0,0.948512,1996-04-14 14:23:59,2016-08-16 22:15:47,Charles Shyer,Steve Martin|Diane Keaton|Martin Short,baby|midlife crisis|confidence|aging|daughter|...,1,True,0


,missing_rate
high_user_rating,0.856860
user_rating_std,0.341224
keywords_list,0.083792
top_cast,0.009910
last_rating_date,0.006276
user_rating_count,0.006276
user_rating_mean,0.006276
first_rating_date,0.006276
genres_list,0.003854
primary_genre,0.003854


### SQL Validation (Path A: Your Own Neon, or Path B: DuckDB)

The project milestone asks for SQL work that supports the workflow. You choose one of two paths:

- **Path A:** your own free Neon database. Upload your tables and query them with SQL.
- **Path B:** DuckDB, which runs SQL directly on pandas DataFrames, so no database server or extra database file is needed.

The notebook `module10_part2_project_sql_options.ipynb` in your Week 9-10 repository explains both paths and has cells marked `# COPY` that you can paste here. The example below uses Path B. The query below checks the movie-level ABT row count and key uniqueness, and a second query rebuilds one aggregation in SQL and compares it with the pandas result.

**Milestone 2 requirement:** include at least one SQL query on your own project data, using Path A or Path B. This cell is a starting example. Extend it with your own query, for example rebuild one aggregation or join from your ABT in SQL and compare the result with pandas. If you use Path A, commit the saved query results in `outputs/sql_results/` (never `.env`), so your notebook runs for the instructor.

In [46]:
import duckdb

# DuckDB can query a pandas DataFrame by its variable name (here: abt).
sql_abt_check = duckdb.sql("""
    SELECT
        COUNT(*) AS abt_rows,
        COUNT(DISTINCT movieId) AS unique_movie_ids,
        CAST(SUM(CASE WHEN high_user_rating IS NULL THEN 1 ELSE 0 END) AS INTEGER) AS target_not_defined_rows
    FROM abt
""").df()
display(sql_abt_check)

# Key check: one row per movieId means the two counts must match.
assert sql_abt_check.loc[0, "abt_rows"] == sql_abt_check.loc[0, "unique_movie_ids"] == len(abt)

# One aggregation in SQL, compared with the same aggregation in pandas.
sql_by_eligibility = duckdb.sql("""
    SELECT
        rating_target_eligible,
        COUNT(*) AS movies,
        AVG(user_rating_mean) AS avg_user_rating
    FROM abt
    WHERE rating_target_eligible IS NOT NULL
    GROUP BY rating_target_eligible
    ORDER BY rating_target_eligible
""").df()

pandas_by_eligibility = (
    abt.groupby("rating_target_eligible")
    .agg(movies=("movieId", "size"), avg_user_rating=("user_rating_mean", "mean"))
    .reset_index()
    .sort_values("rating_target_eligible")
    .reset_index(drop=True)
)

display(sql_by_eligibility)
print("Counts match pandas:", (sql_by_eligibility["movies"].to_numpy() == pandas_by_eligibility["movies"].to_numpy()).all())
print("Averages match pandas:", np.allclose(
    sql_by_eligibility["avg_user_rating"].astype(float), pandas_by_eligibility["avg_user_rating"].astype(float), equal_nan=True
))

,abt_rows,unique_movie_ids,target_not_defined_rows
0,9082,9082,7782


,rating_target_eligible,movies,avg_user_rating
0,False,7782,3.244213
1,True,1300,3.563490


Counts match pandas: True
Averages match pandas: True


### Data Dictionary Starter

Complete the text fields for every column you keep in the final ABT. Remove unused columns before finalizing the dictionary so the documentation and exported ABT stay aligned.


In [49]:
data_dictionary = pd.DataFrame({
    "column_name": abt.columns,
    "data_type": [str(abt[column].dtype) for column in abt.columns],
})

# Hint: use .loc[] to document one column at a time, following the Week 11 pattern.
data_dictionary.head()


,column_name,data_type
0,movieId,int64
1,tmdbId,int64
2,imdb_id,str
3,title,str
4,original_title,str


## Milestone 3 Starter: Prepared Dataset and Initial Model Application

Choose a target/outcome and features. Start simple. Your first model should be correct and interpretable before it becomes complex.

Modeling reminder:

- If you model `high_user_rating`, use only rows where that nullable target is defined; these movies have at least 20 ratings.
- State when the prediction is meant to occur. For a pre-release question, exclude post-release fields such as revenue, popularity, vote count, rating summaries, and rating dates.
- Do not interpret a rating model as proving artistic quality or causal effects.


### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 3 written workspace

Use this cell to document the prepared dataset and initial model/analysis required by the Canvas Milestone 3 page.

- Target/outcome:
- Feature variables:
- Rows included/excluded and why:
- Train/test split or evaluation plan:
- Baseline model or analysis:
- Initial metrics/results:
- Interpretation limits:



In [ ]:
# Milestone 3 code workspace
# Suggested sequence:
# 1. State the prediction or analysis timing in markdown.
# 2. Filter to rows where your target is defined.
# 3. Select only features available at that time.
# 4. Check class balance or the target distribution.
# 5. Create a reproducible train/test split when modeling.
# 6. Fit a simple baseline and evaluate it on test data.


In [ ]:
# Example placeholder. Replace with your chosen target and feature set.
# target = 'your_target_column'
# features = ['feature_1', 'feature_2']
# model_df = abt[[target] + features].dropna()

abt.describe(include='all').T.head(25)

## Final Project Starter: Interpretation, Ethics, and Recommendations

Use this section to connect your analysis back to a business or research decision.

Address:

- Main finding
- What the model/analysis can and cannot support
- Limitations
- Ethics or responsible use concerns
- 2-3 cautious recommendations

### Read the Canvas final submission page first

Before completing this section, read the Canvas final submission assignment page. The Canvas page is the authoritative checklist for the final notebook, HTML report, required files, and submission format. Use this section to organize your final evidence and interpretation.



### Final report written workspace

Use this cell to draft the final interpretation required by the Canvas final submission page.

- Main finding:
- Evidence supporting the finding:
- Business or research implication:
- Limitations:
- Ethics/responsible use concerns:
- Recommendations:
- What you would improve with more time:



In [ ]:
# Final project code workspace
# Save final datasets, figures, or tables here when they are ready.
# Required filenames (final ABT and data dictionary):
# final_abt_path = OUTPUT_DIR / "movies_final_abt.csv"
# dictionary_path = OUTPUT_DIR / "movies_data_dictionary.csv"
# Suggested filename for model metrics:
# metrics_path = OUTPUT_DIR / "movies_model_metrics.csv"
# HTML report: after Restart & Run All, export with the nbconvert command in README.md
# (it creates reports/movies_project_starter.html).


In [ ]:
# Required outputs: save your final ABT and data dictionary with these exact file names.
# Uncomment these lines when your ABT and documentation are complete.
# abt.to_csv(OUTPUT_DIR / "movies_final_abt.csv", index=False)
# data_dictionary.to_csv(OUTPUT_DIR / "movies_data_dictionary.csv", index=False)

# Before final submission, restart the kernel, run all cells, and export the notebook to HTML
# (see "Export the HTML Report" in README.md).


### DuckDB on the Exported ABT File

DuckDB can also query a CSV file directly by its path, without loading it into pandas first. After you save the final ABT above, this check confirms that the exported file has the same row count and unique keys as the `abt` DataFrame.

In [ ]:
final_abt_file = OUTPUT_DIR / "movies_final_abt.csv"

if final_abt_file.exists():
    csv_abt_check = duckdb.sql(f"""
        SELECT
            COUNT(*) AS file_rows,
            COUNT(DISTINCT movieId) AS unique_keys
        FROM read_csv('{final_abt_file.as_posix()}')
    """).df()
    display(csv_abt_check)
    print("Exported file matches abt:", csv_abt_check.loc[0, "file_rows"] == len(abt) == abt["movieId"].nunique())
else:
    print("Export the final ABT first (uncomment the to_csv line above), then rerun this cell.")

## AI Use Disclosure

Keep this cell at the end of your final notebook. State which AI tools you used and what you used them for (for example, debugging an error or explaining a function). If you did not use any AI tools, write "None".

**AI tools used and purpose:** Type your answer here.